# MGMT298D: Science and Strategy of AI
## Week 3: Clustering & Recommendations
### UCLA Anderson School of Management

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

## Load Data

In [ ]:
url = 'https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/netflix_ratings.csv'
raw = pd.read_csv(url)

# Pivot from long format (userId, title, rating) to wide format (users × movies)
df = raw.pivot_table(index='userId', columns='title', values='rating')

n_users = df.shape[0]
n_movies = df.shape[1]
print(f'Users: {n_users}')
print(f'Movies: {n_movies}')
print(f'\nFirst 5 rows (top movies):')
print(df.head())

## Prepare Feature Matrix

In [ ]:
# Impute missing values with column means and standardize
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Data is now standardized (mean=0, std=1)')

## Elbow Method & Silhouette Analysis

In [ ]:
# Test k from 2 to 10
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Find optimal k by silhouette score
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f'Optimal k (by silhouette): {optimal_k}')
print(f'Silhouette scores: {[round(s, 3) for s in silhouette_scores]}')

In [ ]:
# Side-by-side elbow and silhouette plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(optimal_k, color='red', linestyle='--', linewidth=2, label=f'Optimal k={optimal_k}')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score Analysis', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Algorithm Comparison

In [ ]:
# Run three clustering algorithms with k=5
k = 5

# KMeans
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)

# Agglomerative Clustering
agglo = AgglomerativeClustering(n_clusters=k, linkage='ward')
labels_agglo = agglo.fit_predict(X_scaled)

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)

# Compute silhouette scores (handle DBSCAN noise points)
sil_kmeans = silhouette_score(X_scaled, labels_kmeans)
sil_agglo = silhouette_score(X_scaled, labels_agglo)

# For DBSCAN, exclude noise points (-1)
mask_dbscan = labels_dbscan != -1
if mask_dbscan.sum() > 0:
    sil_dbscan = silhouette_score(X_scaled[mask_dbscan], labels_dbscan[mask_dbscan])
else:
    sil_dbscan = np.nan

# Create comparison table
comparison = pd.DataFrame({
    'Algorithm': ['KMeans', 'Agglomerative', 'DBSCAN'],
    'Clusters': [len(set(labels_kmeans)), len(set(labels_agglo)), len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)],
    'Silhouette Score': [round(sil_kmeans, 4), round(sil_agglo, 4), round(sil_dbscan, 4)]
})

print(comparison.to_string(index=False))

## Cluster Profiles (KMeans)

In [ ]:
# Use KMeans labels to analyze cluster profiles
df_with_clusters = df.copy()
df_with_clusters['Cluster'] = labels_kmeans

# For each cluster, show top 5 highest-rated movies
for cluster_id in range(k):
    cluster_users = df_with_clusters[df_with_clusters['Cluster'] == cluster_id]
    cluster_size = len(cluster_users)
    
    avg_ratings = cluster_users.drop('Cluster', axis=1).mean()
    top_5_movies = avg_ratings.nlargest(5)
    
    print(f'\nCluster {cluster_id}: {cluster_size} users')
    print('Top 5 movies:')
    for i, (movie, rating) in enumerate(top_5_movies.items(), 1):
        print(f'  {i}. {movie}: {rating:.2f}')

## Cluster Size Distribution

In [ ]:
# Bar chart of cluster sizes
cluster_sizes = pd.Series(labels_kmeans).value_counts().sort_index()

plt.figure(figsize=(10, 6))
plt.bar(cluster_sizes.index, cluster_sizes.values, color='steelblue', edgecolor='black', alpha=0.8)
plt.xlabel('Cluster ID', fontsize=11)
plt.ylabel('Number of Users', fontsize=11)
plt.title('Users per Cluster (KMeans, k=5)', fontsize=12, fontweight='bold')
plt.xticks(range(k))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## KNN Collaborative Filtering

In [ ]:
# Build KNN model with cosine metric
knn_model = NearestNeighbors(n_neighbors=10, metric='cosine')
knn_model.fit(X_scaled)

# Example 1: Predict rating for user 0, movie 0
user_idx = 0
movie_idx = 0

# Find 10 nearest neighbors
distances, indices = knn_model.kneighbors(X_scaled[user_idx:user_idx+1])

# Get their ratings for the movie
neighbor_ratings = X_imputed[indices[0], movie_idx]

# Predict as weighted average (inverse of cosine distance)
weights = 1 - distances[0]
predicted_rating = np.average(neighbor_ratings, weights=weights)
actual_rating = X_imputed[user_idx, movie_idx]

print(f'User {user_idx}, Movie {movie_idx}:')
print(f'  Actual rating: {actual_rating:.2f}')
print(f'  Predicted rating (KNN): {predicted_rating:.2f}')
print(f'  Error: {abs(actual_rating - predicted_rating):.2f}')

In [ ]:
# Find a user's lowest-rated movie and predict what similar users rated it
user_idx = 1
user_ratings = X_imputed[user_idx]

# Find lowest-rated movie (non-zero)
rated_movies = np.where(user_ratings != 0)[0]
if len(rated_movies) > 0:
    lowest_rated_idx = rated_movies[np.argmin(user_ratings[rated_movies])]
    
    # Find neighbors
    distances, indices = knn_model.kneighbors(X_scaled[user_idx:user_idx+1])
    
    # Get their ratings for this movie
    neighbor_ratings = X_imputed[indices[0], lowest_rated_idx]
    weights = 1 - distances[0]
    predicted_rating = np.average(neighbor_ratings, weights=weights)
    actual_rating = X_imputed[user_idx, lowest_rated_idx]
    
    print(f'User {user_idx} recommendation analysis:')
    print(f'  Their lowest-rated movie: Movie {lowest_rated_idx}')
    print(f'  Their rating: {actual_rating:.2f}')
    print(f'  Similar users rated it: {predicted_rating:.2f} (avg)')
    print(f'  Suggests: Try this movie again or re-evaluate!')

## PCA Cluster Visualization

In [ ]:
# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
var_pc1 = round(explained_var[0] * 100, 1)
var_pc2 = round(explained_var[1] * 100, 1)

# Scatter plot colored by cluster
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_kmeans, cmap='viridis', 
                      alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC1 ({var_pc1}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({var_pc2}% variance)', fontsize=11)
plt.title('KMeans Clusters in PCA Space', fontsize=12, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cluster Preferences Heatmap

In [ ]:
# Create heatmap of average ratings per cluster for first 10 movies
n_movies_heatmap = min(10, df.shape[1])
movies_subset = df.columns[:n_movies_heatmap]

cluster_profiles = []
for cluster_id in range(k):
    cluster_users = df_with_clusters[df_with_clusters['Cluster'] == cluster_id]
    avg_ratings = cluster_users[movies_subset].mean()
    cluster_profiles.append(avg_ratings.values)

cluster_matrix = np.array(cluster_profiles)

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            xticklabels=[m[:15] for m in movies_subset],
            yticklabels=[f'Cluster {i}' for i in range(k)],
            cbar_kws={'label': 'Avg Rating'}, linewidths=0.5)
plt.title('Average Ratings by Cluster (First 10 Movies)', fontsize=12, fontweight='bold')
plt.ylabel('Cluster', fontsize=11)
plt.xlabel('Movie', fontsize=11)
plt.tight_layout()
plt.show()